# 04 — Prepare CNN Windows
# Giai đoạn 1 — Mục 1.5 — Chuẩn bị sliding windows cho CNN raw và envelope
# 
 **Đầu ra**:
- `outputs/tables/windows_cnn_raw.parquet`
- `outputs/tables/windows_cnn_env.parquet`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
from common import io_utils, dsp, features, features_full, config as cfg

In [2]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

- 1. Đọc manifest ĐÃ LÀM SẠCH và cấu hình bandpass

In [3]:
manifest_clean = pd.read_csv(TABLES_DIR / "manifest_clean.csv")

with open(TABLES_DIR / "bandpass_config.json") as f:
    bandpass_cfg = json.load(f)
BAND_HZ = tuple(bandpass_cfg["band_hz"])
LP_CUTOFF_HZ = bandpass_cfg["lp_cutoff_hz"]

# Lấy hết từ common/config.py: bảng đặc trưng MLP (notebook 04) và cửa sổ CNN
# (notebook này) phải dùng CÙNG cửa sổ/stride/warmup, nếu không hai nhánh sẽ
# được chia tập khác nhau và RQ3/RQ4 mất hiệu lực.
TARGET_FS       = cfg.SCOPE["sampling_rate_hz"]
WINDOW_SIZE_RAW = cfg.WINDOW_SIZE_RAW
STRIDE_RAW      = cfg.STRIDE_RAW
ENV_DECIM       = cfg.ENV_DECIM
FS_ENV          = TARGET_FS / ENV_DECIM
WINDOW_SIZE_ENV = cfg.WINDOW_SIZE_ENV
STRIDE_ENV      = cfg.STRIDE_ENV
WARMUP          = cfg.WARMUP_SAMPLES
FS_COL = "resolved_sample_rate_hz" if "resolved_sample_rate_hz" in manifest_clean.columns else "fs"

# Tên nhánh -> tên phương pháp chuẩn trong dsp.py (giống notebook 04)
ENV_METHODS = {
    "squarelaw": "square_law",
    "hilbert":   "hilbert_fir",
    "hybrid":    "hybrid",
}

print(f"raw: {WINDOW_SIZE_RAW} mẫu @{TARGET_FS}Hz = {1000 * WINDOW_SIZE_RAW / TARGET_FS:.2f} ms")
print(f"env: {WINDOW_SIZE_ENV} mẫu @{FS_ENV:.0f}Hz = {1000 * WINDOW_SIZE_ENV / FS_ENV:.2f} ms "
      f"(giảm mẫu 1/{ENV_DECIM})")


raw: 2048 mẫu @12000Hz = 170.67 ms
env: 1024 mẫu @6000Hz = 170.67 ms (giảm mẫu 1/2)


- Prepare CNN Windows

In [4]:
all_windows_raw = []
all_windows_env = {nm: [] for nm in ENV_METHODS}

for _, row in manifest_clean.iterrows():
    if row["label"] is None or pd.isna(row["label"]):
        continue

    # Dùng đúng hàm đọc + resample của bảng đặc trưng MLP thay vì tự gọi
    # resample_poly ở đây: hai nhánh của RQ3 (MLP và CNN 1D) không thể lệch fs
    # hay lệch cách xử lý Normal baseline 48kHz.
    x = io_utils.load_de_signal_resampled(Path(row["file_path"]), row[FS_COL], TARGET_FS)

    # PHẢI dùng đúng make_file_id() của notebook 04. file_id vừa là khóa chia
    # tập của File-based Split/LOLO, vừa là khóa join giữa bảng MLP và bảng
    # CNN. Chuỗi tự ghép trước đây ("IR_1_21.0_209.mat") không join được với
    # "209" của bảng MLP, nên không kiểm chứng được hai nhánh cùng phân hoạch.
    file_id = features_full.make_file_id(row["file_path"])
    meta = {"load_hp": row["load_hp"],
            "fault_diameter_mils": row.get("fault_diameter_mils")}

    w_raw = features.make_sliding_windows(x, WINDOW_SIZE_RAW, overlap_ratio=0.5,
                                          file_id=file_id, label=row["label"],
                                          warmup_samples=WARMUP)
    all_windows_raw.append(w_raw.assign(**meta))

    # Cả ba nhánh envelope đi qua dsp.envelope_by_method() nên dùng CHUNG bậc
    # lọc, lp_cutoff và cách khử DC. Bản trước gọi dsp.hilbert_envelope()
    # (FFT + filtfilt, zero-phase) cạnh hai nhánh nhân quả, nên CNN nhánh
    # Hilbert được lợi kép: không trễ pha và biên độ dải chắn bị bình phương
    # do lọc hai lượt. Khi đó RQ3 không còn đo được cái nó muốn đo.
    for nm, method in ENV_METHODS.items():
        # ĐƯỜNG BAO GỐC, không bật take_sqrt - giống notebook 04. Square-Law
        # trả LP(x²), hai phương pháp kia trả |A|.
        #
        # CNN 1D nhận TRỰC TIẾP mẫu envelope nên nhạy với chênh lệch dải động
        # hơn MLP. Vì vậy lớp chuẩn hóa đầu vào (Normalization/BatchNorm hoặc
        # chuẩn hóa theo cửa sổ) phải được fit RIÊNG cho từng phương pháp ở
        # giai đoạn 2; tuyệt đối không dùng chung một hằng số chuẩn hóa cho cả
        # 3 nhánh, vì khi đó nhánh squarelaw thua do lệch thang chứ không phải
        # do chất lượng đường bao.
        env = dsp.envelope_by_method(
            x, TARGET_FS, band=BAND_HZ, method=method, lp_cutoff=LP_CUTOFF_HZ)
        # Giảm mẫu 1/2 TRƯỚC khi cắt cửa sổ: 1024 mẫu @6kHz = 170.67 ms, đúng
        # bằng 2048 mẫu @12kHz của nhánh raw. Không giảm mẫu thì nhánh env chỉ
        # thấy 85.33 ms (13.8 chu kỳ BPFI so với 27.7) và có gấp đôi số cửa sổ
        # -> RQ4/H4 đang so hai thứ khác nhau cả về thời lượng lẫn cỡ mẫu.
        # Envelope đã qua lowpass LP_CUTOFF_HZ << Nyquist mới (3000 Hz) nên
        # không cần bộ lọc chống chồng phổ bổ sung.
        env_d, _ = dsp.decimate_envelope(env, TARGET_FS, ENV_DECIM, LP_CUTOFF_HZ)
        w_env = features.make_sliding_windows(env_d, WINDOW_SIZE_ENV, overlap_ratio=0.5,
                                              file_id=file_id, label=row["label"],
                                              warmup_samples=WARMUP // ENV_DECIM)
        all_windows_env[nm].append(w_env.assign(dsp_method=nm, **meta))


- 3. Kết xuất dữ liệu

In [5]:
windows_raw_df = pd.concat(all_windows_raw, ignore_index=True)
windows_env_df = {nm: pd.concat(lst, ignore_index=True)
                  for nm, lst in all_windows_env.items()}

windows_raw_df.to_parquet(TABLES_DIR / "windows_cnn_raw.parquet")
for nm, df_env in windows_env_df.items():
    df_env.to_parquet(TABLES_DIR / f"windows_cnn_env_{nm}.parquet")

# Alias tương thích code cũ: mặc định envelope = Square-Law
windows_env_df["squarelaw"].to_parquet(TABLES_DIR / "windows_cnn_env.parquet")

# Cấu hình đầu vào CNN để giai đoạn 2 không phải đoán lại fs/chiều dài
cnn_cfg = {
    "fs_raw_hz": int(TARGET_FS), "window_size_raw": WINDOW_SIZE_RAW,
    "stride_raw": STRIDE_RAW, "fs_env_hz": FS_ENV, "env_decim": ENV_DECIM,
    "window_size_env": WINDOW_SIZE_ENV, "stride_env": STRIDE_ENV,
    "window_duration_ms": round(1000 * WINDOW_SIZE_RAW / TARGET_FS, 3),
    "warmup_samples": WARMUP, "band_hz": list(BAND_HZ),
    "lp_cutoff_hz": LP_CUTOFF_HZ, "env_methods": ENV_METHODS,
}
with open(TABLES_DIR / "windows_cnn_config.json", "w") as f:
    json.dump(cnn_cfg, f, indent=2, ensure_ascii=False)

# --- CHỐT LẠI CÁC HỢP ĐỒNG ------------------------------------------------
keys_raw = set(map(tuple, windows_raw_df[["file_id", "window_idx"]].values))
for tag, df_chk in [("raw", windows_raw_df)] + list(windows_env_df.items()):
    assert "window_idx" in df_chk.columns, (
        f"{tag}: thiếu cột window_idx - common/features.py chưa cập nhật."
    )
    per_file = df_chk.groupby("file_id")["window_idx"].nunique()
    assert per_file.min() > 1, f"{tag}: file_id bị gắn chỉ số cửa sổ, sẽ rò rỉ dữ liệu."
    # Sau khi giảm mẫu 1/2, cửa sổ thứ k của nhánh env phủ ĐÚNG khoảng thời
    # gian của cửa sổ thứ k nhánh raw -> hai tập khóa phải trùng khít.
    assert set(map(tuple, df_chk[["file_id", "window_idx"]].values)) == keys_raw, (
        f"{tag}: tập (file_id, window_idx) lệch nhánh raw - hai nhánh CNN sẽ "
        f"không so sánh được theo cặp."
    )
    print(f"{tag:10s}: {len(df_chk):>6} cửa sổ | {df_chk['file_id'].nunique():>3} file gốc "
          f"| {per_file.min()}-{per_file.max()} cửa sổ/file")

# Khớp với bảng đặc trưng MLP của notebook 04 -> cùng một phân hoạch train/test
mlp_path = TABLES_DIR / "features_mlp.parquet"
if mlp_path.exists():
    mlp_keys = set(map(tuple, pd.read_parquet(mlp_path)[["file_id", "window_idx"]].values))
    assert mlp_keys == keys_raw, (
        "Khóa (file_id, window_idx) của CNN lệch bảng MLP -> hai nhánh sẽ được "
        "chia tập khác nhau. Chạy lại notebook 04 và 05 với cùng cấu hình."
    )
    print(f"Khớp bảng MLP: {len(mlp_keys)} cặp (file_id, window_idx).")
else:
    print("CHƯA có features_mlp.parquet - chạy notebook 04 trước để đối chiếu.")


raw       :   4623 cửa sổ |  40 file gốc | 115-117 cửa sổ/file
squarelaw :   4623 cửa sổ |  40 file gốc | 115-117 cửa sổ/file
hilbert   :   4623 cửa sổ |  40 file gốc | 115-117 cửa sổ/file
hybrid    :   4623 cửa sổ |  40 file gốc | 115-117 cửa sổ/file
Khớp bảng MLP: 4623 cặp (file_id, window_idx).
